In [1]:
# !pip install -r requirements.txt
# Install packages
!pip install -q langchain==0.3.15 langgraph==0.2.45 langchain-core>=0.3.31 openai==1.12.0 statsmodels prophet pandas numpy scikit-learn matplotlib seaborn

# Now paste and run the FIXED agent code
# I'll give you the complete working code in the next message

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.


In [1]:
#!/usr/bin/env python3
"""
TimeSeriesForecaster Agent - Production Grade
A LangGraph-based agent for automated time-series forecasting with ARIMA, SARIMA, and Prophet models.
"""

import os
import sys
import json
import logging
import warnings
import pickle
from datetime import datetime, timedelta
from typing import TypedDict, Literal, Optional, Dict, List, Any, Tuple
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

from langgraph.graph import StateGraph, END

# Note: OpenAI imported lazily in get_openai_client() to avoid initialization issues

# Suppress warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('timeseries_forecaster.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# ============================================================================
# CUSTOM EXCEPTIONS
# ============================================================================

class TimeSeriesError(Exception):
    """Base exception for time series operations"""
    pass

class DataValidationError(TimeSeriesError):
    """Exception raised for data validation errors"""
    pass

class ModelTrainingError(TimeSeriesError):
    """Exception raised for model training errors"""
    pass

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'min_observations': 50,
    'max_missing_pct': 0.5,
    'train_test_split': 0.8,
    'max_model_retries': 3,
    'confidence_levels': [80, 95],
    'default_horizon': 30,
    'openai_model': 'gpt-4o-mini',
    'temperature': 0.7,
    'max_tokens': 2000,
    'max_retry_attempts': 3,
    'figure_size': (14, 8),
    'dpi': 100
}

# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================

def setup_environment() -> str:
    """Setup environment and retrieve OpenAI API key"""
    api_key = None

    # Try Google Colab first
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENAI_API_KEY')
        logger.info("Running in Google Colab environment")
        return api_key
    except ImportError:
        pass
    except Exception as e:
        logger.warning(f"Colab userdata access failed: {e}")

    # Try environment variable
    api_key = os.getenv('OPENAI_API_KEY')
    if api_key:
        logger.info("API key loaded from environment variable")
        return api_key

    # Try .env file
    try:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.getenv('OPENAI_API_KEY')
        if api_key:
            logger.info("API key loaded from .env file")
            return api_key
    except ImportError:
        pass

    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not found. Please set it in:\n"
            "- Google Colab: Secrets (key icon)\n"
            "- Local: .env file or environment variable"
        )

    return api_key

# Initialize OpenAI client - deferred to avoid import-time errors
OPENAI_API_KEY = None
openai_client = None

def get_openai_client():
    """Get or create OpenAI client with proper error handling"""
    global openai_client, OPENAI_API_KEY

    if openai_client is None:
        try:
            if OPENAI_API_KEY is None:
                OPENAI_API_KEY = setup_environment()

            # Import here to avoid issues
            from openai import OpenAI

            # Create client with minimal parameters
            openai_client = OpenAI(
                api_key=OPENAI_API_KEY,
                timeout=60.0,
                max_retries=2
            )
            logger.info("OpenAI client initialized successfully")

        except Exception as e:
            logger.error(f"Failed to initialize OpenAI client: {e}")
            raise EnvironmentError(f"OpenAI client initialization failed: {e}")

    return openai_client

# ============================================================================
# STATE SCHEMA
# ============================================================================

class TimeSeriesState(TypedDict):
    # Input
    raw_data: Optional[pd.DataFrame]
    filepath: Optional[str]
    date_column: Optional[str]
    value_column: Optional[str]
    forecast_horizon: int
    frequency: Optional[str]

    # Analysis
    data_profile: Optional[Dict]
    trend_analysis: Optional[Dict]
    seasonality_analysis: Optional[Dict]
    stationarity_test: Optional[Dict]
    decomposition: Optional[Dict]

    # Modeling
    recommended_models: Optional[List[Dict]]
    selected_model_type: Optional[str]
    model_parameters: Optional[Dict]
    train_data: Optional[pd.Series]
    test_data: Optional[pd.Series]
    trained_model: Optional[Any]

    # Evaluation
    validation_metrics: Optional[Dict]
    residual_analysis: Optional[Dict]

    # Forecasting
    forecast_values: Optional[pd.Series]
    confidence_intervals: Optional[Dict]
    forecast_dates: Optional[pd.DatetimeIndex]

    # Insights
    pattern_insights: Optional[List[str]]
    forecast_insights: Optional[List[str]]
    recommendations: Optional[List[str]]

    # Output
    plots: Optional[Dict[str, str]]
    processing_stage: str
    errors: List[str]
    warnings: List[str]
    retry_count: int

# ============================================================================
# FILE LOADING UTILITIES
# ============================================================================

def load_timeseries_data(filepath: str) -> pd.DataFrame:
    """
    Load time-series data from various file formats.

    Args:
        filepath: Path to the data file

    Returns:
        DataFrame containing the time-series data

    Raises:
        DataValidationError: If file cannot be loaded
    """
    try:
        file_path = Path(filepath)

        if not file_path.exists():
            raise DataValidationError(f"File not found: {filepath}")

        file_extension = file_path.suffix.lower()

        if file_extension == '.csv':
            df = pd.read_csv(filepath)
            logger.info(f"Loaded CSV file with {len(df)} rows")

        elif file_extension in ['.xlsx', '.xls']:
            df = pd.read_excel(filepath, sheet_name=0)
            logger.info(f"Loaded Excel file with {len(df)} rows")

        elif file_extension == '.json':
            df = pd.read_json(filepath)
            logger.info(f"Loaded JSON file with {len(df)} rows")

        else:
            raise DataValidationError(
                f"Unsupported file format: {file_extension}. "
                "Supported formats: .csv, .xlsx, .xls, .json"
            )

        if df.empty:
            raise DataValidationError("Loaded file is empty")

        return df

    except Exception as e:
        logger.error(f"Error loading file: {str(e)}")
        raise DataValidationError(f"Failed to load file: {str(e)}")

def get_file_from_user() -> str:
    """
    Get file path from user with file selection dialog or manual input.

    Returns:
        Path to the selected file
    """
    # Try to use file dialog
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)

        filepath = filedialog.askopenfilename(
            title='Select Time Series Data File',
            filetypes=[
                ('CSV files', '*.csv'),
                ('Excel files', '*.xlsx *.xls'),
                ('JSON files', '*.json'),
                ('All files', '*.*')
            ]
        )

        root.destroy()

        if filepath:
            logger.info(f"File selected via dialog: {filepath}")
            return filepath

    except Exception as e:
        logger.warning(f"File dialog not available: {e}")

    # Fallback to manual input
    print("\n" + "="*80)
    print("FILE SELECTION")
    print("="*80)
    filepath = input("\nEnter the path to your time-series data file: ").strip()

    # Remove quotes if present
    filepath = filepath.strip('"').strip("'")

    return filepath

# ============================================================================
# LLM INTEGRATION
# ============================================================================

def call_llm_structured(
    system_prompt: str,
    user_prompt: str,
    response_format: Optional[Dict] = None
) -> Dict:
    """
    Call OpenAI API with structured output.

    Args:
        system_prompt: System instructions
        user_prompt: User query
        response_format: JSON schema for structured output

    Returns:
        Parsed JSON response
    """
    try:
        client = get_openai_client()

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        if response_format:
            response = client.chat.completions.create(
                model=CONFIG['openai_model'],
                messages=messages,
                temperature=CONFIG['temperature'],
                max_tokens=CONFIG['max_tokens'],
                response_format={"type": "json_object"}
            )
        else:
            response = client.chat.completions.create(
                model=CONFIG['openai_model'],
                messages=messages,
                temperature=CONFIG['temperature'],
                max_tokens=CONFIG['max_tokens']
            )

        content = response.choices[0].message.content

        if response_format:
            return json.loads(content)
        else:
            return {"content": content}

    except Exception as e:
        logger.error(f"LLM API call failed: {str(e)}")
        raise TimeSeriesError(f"LLM call failed: {str(e)}")

# ============================================================================
# TIME SERIES ANALYSIS FUNCTIONS
# ============================================================================

def detect_trend(series: pd.Series) -> Dict:
    """Detect trend in time series"""
    try:
        # Linear regression for trend
        x = np.arange(len(series))
        y = series.values

        # Remove NaN values
        mask = ~np.isnan(y)
        x_clean = x[mask]
        y_clean = y[mask]

        if len(x_clean) < 2:
            return {"trend_detected": False, "trend_strength": 0}

        coefficients = np.polyfit(x_clean, y_clean, 1)
        slope = coefficients[0]

        # Calculate R-squared
        y_pred = np.polyval(coefficients, x_clean)
        ss_res = np.sum((y_clean - y_pred) ** 2)
        ss_tot = np.sum((y_clean - np.mean(y_clean)) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

        trend_type = "increasing" if slope > 0 else "decreasing" if slope < 0 else "flat"

        return {
            "trend_detected": abs(slope) > 0.01,
            "trend_type": trend_type,
            "slope": float(slope),
            "trend_strength": float(r_squared)
        }

    except Exception as e:
        logger.error(f"Trend detection failed: {e}")
        return {"trend_detected": False, "trend_strength": 0}

def detect_seasonality(series: pd.Series, frequency: int = None) -> Dict:
    """Detect seasonality patterns"""
    try:
        if frequency is None:
            # Auto-detect frequency
            if len(series) >= 730:
                frequency = 365  # Yearly
            elif len(series) >= 60:
                frequency = 7  # Weekly
            else:
                return {"seasonality_detected": False}

        if len(series) < 2 * frequency:
            return {"seasonality_detected": False}

        # Autocorrelation at seasonal lag
        series_clean = series.dropna()
        if len(series_clean) < frequency + 1:
            return {"seasonality_detected": False}

        autocorr = series_clean.autocorr(lag=frequency)

        seasonality_detected = abs(autocorr) > 0.3

        return {
            "seasonality_detected": seasonality_detected,
            "frequency": frequency,
            "autocorr": float(autocorr) if not np.isnan(autocorr) else 0.0,
            "strength": abs(float(autocorr)) if not np.isnan(autocorr) else 0.0
        }

    except Exception as e:
        logger.error(f"Seasonality detection failed: {e}")
        return {"seasonality_detected": False}

def test_stationarity(series: pd.Series) -> Dict:
    """Test time series for stationarity using ADF and KPSS tests"""
    try:
        series_clean = series.dropna()

        if len(series_clean) < 10:
            return {"is_stationary": False, "confidence": "low"}

        # Augmented Dickey-Fuller test
        adf_result = adfuller(series_clean, autolag='AIC')

        # KPSS test
        try:
            kpss_result = kpss(series_clean, regression='c', nlags='auto')
            kpss_p_value = kpss_result[1]
        except:
            kpss_p_value = None

        adf_statistic = adf_result[0]
        adf_p_value = adf_result[1]
        adf_critical_values = adf_result[4]

        # Stationary if ADF rejects null (p < 0.05) and KPSS accepts null (p > 0.05)
        is_stationary = adf_p_value < 0.05

        if kpss_p_value is not None:
            is_stationary = is_stationary and kpss_p_value > 0.05

        return {
            "is_stationary": is_stationary,
            "adf_statistic": float(adf_statistic),
            "adf_p_value": float(adf_p_value),
            "adf_critical_values": {k: float(v) for k, v in adf_critical_values.items()},
            "kpss_p_value": float(kpss_p_value) if kpss_p_value else None,
            "confidence": "high" if adf_p_value < 0.01 else "medium"
        }

    except Exception as e:
        logger.error(f"Stationarity test failed: {e}")
        return {"is_stationary": False, "confidence": "low"}

def decompose_series(series: pd.Series, frequency: int = None) -> Dict:
    """Decompose time series into trend, seasonal, and residual components"""
    try:
        if frequency is None:
            frequency = 7 if len(series) < 730 else 365

        if len(series) < 2 * frequency:
            return {"decomposition_successful": False}

        decomposition = seasonal_decompose(
            series.dropna(),
            model='additive',
            period=frequency,
            extrapolate_trend='freq'
        )

        return {
            "decomposition_successful": True,
            "trend": decomposition.trend,
            "seasonal": decomposition.seasonal,
            "residual": decomposition.resid,
            "frequency": frequency
        }

    except Exception as e:
        logger.error(f"Decomposition failed: {e}")
        return {"decomposition_successful": False}

def detect_outliers(series: pd.Series) -> Dict:
    """Detect outliers using IQR method"""
    try:
        series_clean = series.dropna()

        Q1 = series_clean.quantile(0.25)
        Q3 = series_clean.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = series_clean[(series_clean < lower_bound) | (series_clean > upper_bound)]

        return {
            "outlier_count": len(outliers),
            "outlier_pct": len(outliers) / len(series_clean) * 100,
            "outlier_indices": outliers.index.tolist(),
            "lower_bound": float(lower_bound),
            "upper_bound": float(upper_bound)
        }

    except Exception as e:
        logger.error(f"Outlier detection failed: {e}")
        return {"outlier_count": 0, "outlier_pct": 0}

# ============================================================================
# FORECASTING MODEL FUNCTIONS
# ============================================================================

def find_optimal_arima_params(series: pd.Series, seasonal: bool = False) -> Tuple:
    """Find optimal ARIMA parameters using grid search"""
    try:
        series_clean = series.dropna()

        # Simple grid search
        best_aic = np.inf
        best_params = (1, 1, 1)

        p_range = range(0, 3)
        d_range = range(0, 2)
        q_range = range(0, 3)

        for p in p_range:
            for d in d_range:
                for q in q_range:
                    try:
                        model = ARIMA(series_clean, order=(p, d, q))
                        fitted = model.fit()

                        if fitted.aic < best_aic:
                            best_aic = fitted.aic
                            best_params = (p, d, q)
                    except:
                        continue

        if seasonal:
            # Add seasonal component
            return best_params + (0, 1, 0, 7)

        return best_params

    except Exception as e:
        logger.error(f"Parameter optimization failed: {e}")
        return (1, 1, 1) if not seasonal else (1, 1, 1, 0, 1, 0, 7)

def train_forecasting_model(
    train_data: pd.Series,
    model_type: str,
    parameters: Dict
) -> Any:
    """Train the selected forecasting model"""
    try:
        train_clean = train_data.dropna()

        if model_type == 'ARIMA':
            order = parameters.get('order', (1, 1, 1))
            model = ARIMA(train_clean, order=order)
            fitted_model = model.fit()

        elif model_type == 'SARIMA':
            order = parameters.get('order', (1, 1, 1))
            seasonal_order = parameters.get('seasonal_order', (0, 1, 0, 7))
            model = SARIMAX(train_clean, order=order, seasonal_order=seasonal_order)
            fitted_model = model.fit(disp=False)

        elif model_type == 'Prophet':
            df_prophet = pd.DataFrame({
                'ds': train_clean.index,
                'y': train_clean.values
            })
            model = Prophet(
                yearly_seasonality=parameters.get('yearly_seasonality', True),
                weekly_seasonality=parameters.get('weekly_seasonality', True),
                daily_seasonality=parameters.get('daily_seasonality', False)
            )
            fitted_model = model.fit(df_prophet)

        elif model_type == 'ExponentialSmoothing':
            model = ExponentialSmoothing(
                train_clean,
                seasonal_periods=parameters.get('seasonal_periods', 7),
                trend=parameters.get('trend', 'add'),
                seasonal=parameters.get('seasonal', 'add')
            )
            fitted_model = model.fit()

        else:
            raise ModelTrainingError(f"Unknown model type: {model_type}")

        logger.info(f"{model_type} model trained successfully")
        return fitted_model

    except Exception as e:
        logger.error(f"Model training failed: {e}")
        raise ModelTrainingError(f"Failed to train {model_type}: {str(e)}")

def generate_forecast(
    model: Any,
    model_type: str,
    horizon: int,
    last_date: pd.Timestamp,
    frequency: str
) -> Tuple[pd.Series, Dict]:
    """Generate forecast with confidence intervals"""
    try:
        if model_type in ['ARIMA', 'SARIMA']:
            forecast_result = model.forecast(steps=horizon)
            forecast_values = pd.Series(forecast_result)

            # Get confidence intervals
            forecast_obj = model.get_forecast(steps=horizon)
            conf_int = forecast_obj.conf_int()

            confidence_intervals = {
                '95_lower': conf_int.iloc[:, 0].values,
                '95_upper': conf_int.iloc[:, 1].values
            }

        elif model_type == 'Prophet':
            future_dates = model.make_future_dataframe(periods=horizon, freq=frequency)
            forecast = model.predict(future_dates)

            forecast_values = pd.Series(
                forecast['yhat'].iloc[-horizon:].values
            )

            confidence_intervals = {
                '95_lower': forecast['yhat_lower'].iloc[-horizon:].values,
                '95_upper': forecast['yhat_upper'].iloc[-horizon:].values
            }

        elif model_type == 'ExponentialSmoothing':
            forecast_values = model.forecast(steps=horizon)

            # Simple confidence intervals (±10%)
            confidence_intervals = {
                '95_lower': forecast_values.values * 0.9,
                '95_upper': forecast_values.values * 1.1
            }

        else:
            raise ModelTrainingError(f"Unknown model type: {model_type}")

        # Generate forecast dates
        if frequency == 'D':
            freq = 'D'
        elif frequency in ['W', 'W-SUN']:
            freq = 'W'
        elif frequency in ['M', 'MS']:
            freq = 'MS'
        else:
            freq = 'D'

        forecast_dates = pd.date_range(
            start=last_date + pd.Timedelta(days=1),
            periods=horizon,
            freq=freq
        )

        forecast_values.index = forecast_dates

        return forecast_values, confidence_intervals

    except Exception as e:
        logger.error(f"Forecast generation failed: {e}")
        raise ModelTrainingError(f"Failed to generate forecast: {str(e)}")

# ============================================================================
# LANGGRAPH NODES
# ============================================================================

def file_loader_node(state: TimeSeriesState) -> TimeSeriesState:
    """Load and parse the time-series data file"""
    logger.info("=" * 80)
    logger.info("STAGE 1: File Loading")
    logger.info("=" * 80)

    try:
        filepath = state['filepath']
        df = load_timeseries_data(filepath)

        state['raw_data'] = df
        state['processing_stage'] = 'file_loaded'

        logger.info(f"✓ File loaded successfully: {len(df)} rows, {len(df.columns)} columns")
        logger.info(f"Columns: {', '.join(df.columns.tolist())}")

    except Exception as e:
        state['errors'].append(f"File loading failed: {str(e)}")
        logger.error(f"✗ File loading failed: {e}")

    return state

def column_identifier_node(state: TimeSeriesState) -> TimeSeriesState:
    """Use LLM to identify date and value columns"""
    logger.info("=" * 80)
    logger.info("STAGE 2: Column Identification")
    logger.info("=" * 80)

    try:
        df = state['raw_data']

        # Prepare column information
        column_info = []
        for col in df.columns:
            sample_values = df[col].dropna().head(5).tolist()
            dtype = str(df[col].dtype)
            column_info.append({
                "name": col,
                "dtype": dtype,
                "sample_values": [str(v) for v in sample_values]
            })

        system_prompt = """You are an expert data analyst specializing in time-series data.
Your task is to identify which columns contain the date/time information and which contain the values to forecast.

Return your response as a JSON object with this exact structure:
{
    "date_column": "column_name",
    "value_column": "column_name",
    "confidence": "high|medium|low",
    "reasoning": "brief explanation"
}"""

        user_prompt = f"""Analyze these columns and identify the date and value columns:

{json.dumps(column_info, indent=2)}

Identify:
1. date_column: The column containing dates/timestamps
2. value_column: The column containing numeric values to forecast

Return only valid JSON."""

        result = call_llm_structured(system_prompt, user_prompt, response_format=True)

        state['date_column'] = result['date_column']
        state['value_column'] = result['value_column']
        state['processing_stage'] = 'columns_identified'

        logger.info(f"✓ Date column: {result['date_column']}")
        logger.info(f"✓ Value column: {result['value_column']}")
        logger.info(f"Confidence: {result.get('confidence', 'unknown')}")

    except Exception as e:
        state['errors'].append(f"Column identification failed: {str(e)}")
        logger.error(f"✗ Column identification failed: {e}")

    return state

def data_validator_node(state: TimeSeriesState) -> TimeSeriesState:
    """Validate and prepare time-series data"""
    logger.info("=" * 80)
    logger.info("STAGE 3: Data Validation")
    logger.info("=" * 80)

    try:
        df = state['raw_data'].copy()
        date_col = state['date_column']
        value_col = state['value_column']

        # Convert date column
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col)
        df = df.set_index(date_col)

        # Extract value series
        series = df[value_col]

        # Validate minimum observations
        if len(series) < CONFIG['min_observations']:
            raise DataValidationError(
                f"Insufficient data: {len(series)} rows (minimum {CONFIG['min_observations']} required)"
            )

        # Check missing values
        missing_pct = series.isna().sum() / len(series)
        if missing_pct > CONFIG['max_missing_pct']:
            raise DataValidationError(
                f"Too many missing values: {missing_pct:.1%} (maximum {CONFIG['max_missing_pct']:.0%} allowed)"
            )

        # Infer frequency
        freq = pd.infer_freq(series.index)
        if freq is None:
            # Estimate based on median difference
            time_diffs = series.index.to_series().diff().dropna()
            median_diff = time_diffs.median()

            if median_diff <= pd.Timedelta(days=1):
                freq = 'D'
            elif median_diff <= pd.Timedelta(days=7):
                freq = 'W'
            else:
                freq = 'MS'

        state['frequency'] = freq
        state['raw_data'] = series
        state['processing_stage'] = 'data_validated'

        # Create data profile
        state['data_profile'] = {
            'total_observations': len(series),
            'date_range': {
                'start': str(series.index.min()),
                'end': str(series.index.max())
            },
            'missing_values': int(series.isna().sum()),
            'missing_pct': float(missing_pct * 100),
            'frequency': freq,
            'statistics': {
                'mean': float(series.mean()),
                'median': float(series.median()),
                'std': float(series.std()),
                'min': float(series.min()),
                'max': float(series.max())
            }
        }

        logger.info(f"✓ Data validated: {len(series)} observations")
        logger.info(f"✓ Date range: {series.index.min()} to {series.index.max()}")
        logger.info(f"✓ Frequency: {freq}")
        logger.info(f"✓ Missing values: {missing_pct:.1%}")

    except Exception as e:
        state['errors'].append(f"Data validation failed: {str(e)}")
        logger.error(f"✗ Data validation failed: {e}")

    return state

def exploratory_analysis_node(state: TimeSeriesState) -> TimeSeriesState:
    """Perform exploratory time-series analysis"""
    logger.info("=" * 80)
    logger.info("STAGE 4: Exploratory Analysis")
    logger.info("=" * 80)

    try:
        series = state['raw_data']

        # Trend detection
        trend_info = detect_trend(series)
        state['trend_analysis'] = trend_info

        logger.info(f"✓ Trend detected: {trend_info.get('trend_detected', False)}")
        if trend_info.get('trend_detected'):
            logger.info(f"  Type: {trend_info.get('trend_type', 'unknown')}")
            logger.info(f"  Strength: {trend_info.get('trend_strength', 0):.3f}")

        # Seasonality detection
        freq_map = {'D': 7, 'W': 52, 'MS': 12, 'M': 12}
        frequency = freq_map.get(state['frequency'], 7)

        seasonality_info = detect_seasonality(series, frequency)
        state['seasonality_analysis'] = seasonality_info

        logger.info(f"✓ Seasonality detected: {seasonality_info.get('seasonality_detected', False)}")
        if seasonality_info.get('seasonality_detected'):
            logger.info(f"  Frequency: {seasonality_info.get('frequency', 0)}")
            logger.info(f"  Strength: {seasonality_info.get('strength', 0):.3f}")

        # Outlier detection
        outlier_info = detect_outliers(series)
        logger.info(f"✓ Outliers found: {outlier_info['outlier_count']} ({outlier_info['outlier_pct']:.1f}%)")

        state['processing_stage'] = 'exploratory_complete'

    except Exception as e:
        state['errors'].append(f"Exploratory analysis failed: {str(e)}")
        logger.error(f"✗ Exploratory analysis failed: {e}")

    return state

def stationarity_test_node(state: TimeSeriesState) -> TimeSeriesState:
    """Test for stationarity and perform decomposition"""
    logger.info("=" * 80)
    logger.info("STAGE 5: Stationarity Testing")
    logger.info("=" * 80)

    try:
        series = state['raw_data']

        # Stationarity test
        stationarity_result = test_stationarity(series)
        state['stationarity_test'] = stationarity_result

        logger.info(f"✓ Stationarity test complete")
        logger.info(f"  Stationary: {stationarity_result['is_stationary']}")
        logger.info(f"  ADF p-value: {stationarity_result.get('adf_p_value', 'N/A')}")

        # Decomposition
        freq_map = {'D': 7, 'W': 52, 'MS': 12, 'M': 12}
        frequency = freq_map.get(state['frequency'], 7)

        decomp_result = decompose_series(series, frequency)
        state['decomposition'] = decomp_result

        if decomp_result.get('decomposition_successful'):
            logger.info(f"✓ Decomposition successful (frequency: {frequency})")

        state['processing_stage'] = 'stationarity_tested'

    except Exception as e:
        state['errors'].append(f"Stationarity testing failed: {str(e)}")
        logger.error(f"✗ Stationarity testing failed: {e}")

    return state

def model_recommendation_node(state: TimeSeriesState) -> TimeSeriesState:
    """Use LLM to recommend forecasting models"""
    logger.info("=" * 80)
    logger.info("STAGE 6: Model Recommendation")
    logger.info("=" * 80)

    try:
        # Prepare analysis summary
        analysis_summary = {
            "data_profile": state['data_profile'],
            "trend": state['trend_analysis'],
            "seasonality": state['seasonality_analysis'],
            "stationarity": {
                "is_stationary": state['stationarity_test']['is_stationary'],
                "adf_p_value": state['stationarity_test'].get('adf_p_value')
            }
        }

        system_prompt = """You are an expert time-series forecasting analyst.
Based on the data characteristics, recommend the most appropriate forecasting models.

Available models:
- ARIMA: For stationary or differenced data without strong seasonality
- SARIMA: For data with seasonal patterns
- Prophet: For data with multiple seasonalities and holidays
- ExponentialSmoothing: For data with trend and/or seasonality

Return your response as a JSON object with this structure:
{
    "recommended_models": [
        {
            "model_type": "model_name",
            "priority": 1,
            "reasoning": "why this model is suitable",
            "expected_performance": "high|medium|low"
        }
    ],
    "primary_recommendation": "model_name"
}"""

        user_prompt = f"""Analyze this time-series data and recommend forecasting models:

{json.dumps(analysis_summary, indent=2, default=str)}

Consider:
1. Trend presence and strength
2. Seasonality patterns
3. Stationarity
4. Data length and frequency

Recommend 2-3 models ranked by suitability. Return only valid JSON."""

        result = call_llm_structured(system_prompt, user_prompt, response_format=True)

        state['recommended_models'] = result['recommended_models']
        state['selected_model_type'] = result['primary_recommendation']
        state['processing_stage'] = 'model_recommended'

        logger.info(f"✓ Primary recommendation: {result['primary_recommendation']}")
        for model in result['recommended_models']:
            logger.info(f"  {model['priority']}. {model['model_type']} - {model['expected_performance']} performance")

    except Exception as e:
        # Fallback to rule-based recommendation
        logger.warning(f"LLM recommendation failed, using rule-based fallback: {e}")

        has_seasonality = state['seasonality_analysis'].get('seasonality_detected', False)
        is_stationary = state['stationarity_test'].get('is_stationary', False)

        if has_seasonality:
            model_type = 'SARIMA'
        elif not is_stationary:
            model_type = 'ARIMA'
        else:
            model_type = 'ExponentialSmoothing'

        state['selected_model_type'] = model_type
        state['recommended_models'] = [{"model_type": model_type, "priority": 1}]

        logger.info(f"✓ Fallback recommendation: {model_type}")

    return state

def model_selection_node(state: TimeSeriesState) -> TimeSeriesState:
    """Select model and determine optimal parameters"""
    logger.info("=" * 80)
    logger.info("STAGE 7: Model Selection & Parameter Tuning")
    logger.info("=" * 80)

    try:
        model_type = state['selected_model_type']
        series = state['raw_data']

        parameters = {}

        if model_type == 'ARIMA':
            order = find_optimal_arima_params(series, seasonal=False)
            parameters['order'] = order
            logger.info(f"✓ ARIMA parameters: p={order[0]}, d={order[1]}, q={order[2]}")

        elif model_type == 'SARIMA':
            seasonal_order = find_optimal_arima_params(series, seasonal=True)
            parameters['order'] = seasonal_order[:3]
            parameters['seasonal_order'] = seasonal_order[3:] if len(seasonal_order) > 3 else (0, 1, 0, 7)
            logger.info(f"✓ SARIMA parameters: {parameters['order']} x {parameters['seasonal_order']}")

        elif model_type == 'Prophet':
            parameters['yearly_seasonality'] = state['seasonality_analysis'].get('seasonality_detected', True)
            parameters['weekly_seasonality'] = True
            parameters['daily_seasonality'] = False
            logger.info(f"✓ Prophet parameters configured")

        elif model_type == 'ExponentialSmoothing':
            freq_map = {'D': 7, 'W': 52, 'MS': 12, 'M': 12}
            parameters['seasonal_periods'] = freq_map.get(state['frequency'], 7)
            parameters['trend'] = 'add' if state['trend_analysis'].get('trend_detected') else None
            parameters['seasonal'] = 'add' if state['seasonality_analysis'].get('seasonality_detected') else None
            logger.info(f"✓ ExponentialSmoothing parameters configured")

        state['model_parameters'] = parameters
        state['processing_stage'] = 'model_selected'

    except Exception as e:
        state['errors'].append(f"Model selection failed: {str(e)}")
        logger.error(f"✗ Model selection failed: {e}")

    return state

def data_splitting_node(state: TimeSeriesState) -> TimeSeriesState:
    """Split data into training and testing sets"""
    logger.info("=" * 80)
    logger.info("STAGE 8: Data Splitting")
    logger.info("=" * 80)

    try:
        series = state['raw_data']
        split_point = int(len(series) * CONFIG['train_test_split'])

        train_data = series.iloc[:split_point]
        test_data = series.iloc[split_point:]

        state['train_data'] = train_data
        state['test_data'] = test_data
        state['processing_stage'] = 'data_split'

        logger.info(f"✓ Training set: {len(train_data)} observations")
        logger.info(f"✓ Testing set: {len(test_data)} observations")

    except Exception as e:
        state['errors'].append(f"Data splitting failed: {str(e)}")
        logger.error(f"✗ Data splitting failed: {e}")

    return state

def model_training_node(state: TimeSeriesState) -> TimeSeriesState:
    """Train the selected forecasting model"""
    logger.info("=" * 80)
    logger.info("STAGE 9: Model Training")
    logger.info("=" * 80)

    try:
        train_data = state['train_data']
        model_type = state['selected_model_type']
        parameters = state['model_parameters']

        logger.info(f"Training {model_type} model...")

        trained_model = train_forecasting_model(train_data, model_type, parameters)

        state['trained_model'] = trained_model
        state['processing_stage'] = 'model_trained'

        logger.info(f"✓ {model_type} model trained successfully")

    except Exception as e:
        state['errors'].append(f"Model training failed: {str(e)}")
        logger.error(f"✗ Model training failed: {e}")

        # Retry with simpler model
        if state['retry_count'] < CONFIG['max_model_retries']:
            state['retry_count'] += 1
            state['selected_model_type'] = 'ARIMA'
            state['model_parameters'] = {'order': (1, 1, 1)}
            logger.warning(f"Retrying with ARIMA (attempt {state['retry_count']})")

    return state

def model_evaluation_node(state: TimeSeriesState) -> TimeSeriesState:
    """Evaluate model performance on test set"""
    logger.info("=" * 80)
    logger.info("STAGE 10: Model Evaluation")
    logger.info("=" * 80)

    try:
        model = state['trained_model']
        test_data = state['test_data']
        model_type = state['selected_model_type']

        # Generate predictions for test set
        if model_type in ['ARIMA', 'SARIMA']:
            predictions = model.forecast(steps=len(test_data))
        elif model_type == 'Prophet':
            future_df = pd.DataFrame({'ds': test_data.index})
            forecast = model.predict(future_df)
            predictions = forecast['yhat'].values
        elif model_type == 'ExponentialSmoothing':
            predictions = model.forecast(steps=len(test_data))

        # Calculate metrics
        mae = mean_absolute_error(test_data, predictions)
        rmse = np.sqrt(mean_squared_error(test_data, predictions))
        mape = mean_absolute_percentage_error(test_data, predictions) * 100

        # Residual analysis
        residuals = test_data.values - predictions

        state['validation_metrics'] = {
            'mae': float(mae),
            'rmse': float(rmse),
            'mape': float(mape)
        }

        state['residual_analysis'] = {
            'mean': float(np.mean(residuals)),
            'std': float(np.std(residuals)),
            'min': float(np.min(residuals)),
            'max': float(np.max(residuals))
        }

        state['processing_stage'] = 'model_evaluated'

        logger.info(f"✓ Model evaluation complete")
        logger.info(f"  MAE: {mae:.2f}")
        logger.info(f"  RMSE: {rmse:.2f}")
        logger.info(f"  MAPE: {mape:.2f}%")

    except Exception as e:
        state['warnings'].append(f"Model evaluation encountered issues: {str(e)}")
        logger.warning(f"⚠ Model evaluation failed: {e}")
        state['processing_stage'] = 'model_evaluated'

    return state

def forecasting_node(state: TimeSeriesState) -> TimeSeriesState:
    """Generate future forecasts"""
    logger.info("=" * 80)
    logger.info("STAGE 11: Forecasting")
    logger.info("=" * 80)

    try:
        model = state['trained_model']
        model_type = state['selected_model_type']
        horizon = state['forecast_horizon']
        series = state['raw_data']
        frequency = state['frequency']

        last_date = series.index[-1]

        forecast_values, confidence_intervals = generate_forecast(
            model, model_type, horizon, last_date, frequency
        )

        state['forecast_values'] = forecast_values
        state['confidence_intervals'] = confidence_intervals
        state['forecast_dates'] = forecast_values.index
        state['processing_stage'] = 'forecast_generated'

        logger.info(f"✓ Forecast generated: {horizon} periods")
        logger.info(f"  Forecast range: {forecast_values.index[0]} to {forecast_values.index[-1]}")
        logger.info(f"  Mean forecast: {forecast_values.mean():.2f}")

    except Exception as e:
        state['errors'].append(f"Forecasting failed: {str(e)}")
        logger.error(f"✗ Forecasting failed: {e}")

    return state

def insights_generation_node(state: TimeSeriesState) -> TimeSeriesState:
    """Generate business insights using LLM"""
    logger.info("=" * 80)
    logger.info("STAGE 12: Insights Generation")
    logger.info("=" * 80)

    try:
        # Prepare comprehensive summary
        summary = {
            "data_profile": state['data_profile'],
            "trend": state['trend_analysis'],
            "seasonality": state['seasonality_analysis'],
            "model_used": state['selected_model_type'],
            "validation_metrics": state.get('validation_metrics', {}),
            "forecast_summary": {
                "horizon": state['forecast_horizon'],
                "mean_forecast": float(state['forecast_values'].mean()),
                "min_forecast": float(state['forecast_values'].min()),
                "max_forecast": float(state['forecast_values'].max())
            }
        }

        system_prompt = """You are an expert business analyst specializing in time-series forecasting.
Generate actionable insights and recommendations based on the analysis and forecast results.

Provide:
1. Pattern insights: Key patterns discovered in historical data
2. Forecast insights: What the forecast indicates for the future
3. Recommendations: Strategic recommendations for decision-making

Return your response as a JSON object:
{
    "pattern_insights": ["insight 1", "insight 2", ...],
    "forecast_insights": ["insight 1", "insight 2", ...],
    "recommendations": ["recommendation 1", "recommendation 2", ...]
}"""

        user_prompt = f"""Analyze this time-series forecast and provide business insights:

{json.dumps(summary, indent=2, default=str)}

Generate:
- 3-5 pattern insights from historical data
- 3-5 forecast insights about future trends
- 3-5 actionable recommendations

Be specific, quantitative, and business-focused. Return only valid JSON."""

        result = call_llm_structured(system_prompt, user_prompt, response_format=True)

        state['pattern_insights'] = result['pattern_insights']
        state['forecast_insights'] = result['forecast_insights']
        state['recommendations'] = result['recommendations']
        state['processing_stage'] = 'insights_generated'

        logger.info(f"✓ Insights generated")
        logger.info(f"  Pattern insights: {len(result['pattern_insights'])}")
        logger.info(f"  Forecast insights: {len(result['forecast_insights'])}")
        logger.info(f"  Recommendations: {len(result['recommendations'])}")

    except Exception as e:
        state['warnings'].append(f"Insights generation failed: {str(e)}")
        logger.warning(f"⚠ Insights generation failed: {e}")

        # Fallback to basic insights
        state['pattern_insights'] = ["Analysis complete - see visualizations for details"]
        state['forecast_insights'] = [f"Forecast generated for {state['forecast_horizon']} periods"]
        state['recommendations'] = ["Review forecast visualizations and validate assumptions"]
        state['processing_stage'] = 'insights_generated'

    return state

def visualization_node(state: TimeSeriesState) -> TimeSeriesState:
    """Generate visualizations"""
    logger.info("=" * 80)
    logger.info("STAGE 13: Visualization")
    logger.info("=" * 80)

    try:
        series = state['raw_data']
        forecast = state['forecast_values']
        conf_int = state['confidence_intervals']

        plots = {}

        # 1. Time series with forecast
        fig, ax = plt.subplots(figsize=CONFIG['figure_size'], dpi=CONFIG['dpi'])

        # Historical data
        ax.plot(series.index, series.values, label='Historical Data', color='#2E86AB', linewidth=2)

        # Forecast
        ax.plot(forecast.index, forecast.values, label='Forecast', color='#A23B72', linewidth=2, linestyle='--')

        # Confidence intervals
        ax.fill_between(
            forecast.index,
            conf_int['95_lower'],
            conf_int['95_upper'],
            alpha=0.2,
            color='#A23B72',
            label='95% Confidence Interval'
        )

        ax.set_xlabel('Date', fontsize=12)
        ax.set_ylabel('Value', fontsize=12)
        ax.set_title('Time Series Forecast', fontsize=14, fontweight='bold')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plot_path = 'forecast_plot.png'
        plt.savefig(plot_path)
        plt.close()
        plots['forecast'] = plot_path

        logger.info(f"✓ Forecast plot saved: {plot_path}")

        # 2. Decomposition plot (if available)
        if state['decomposition'].get('decomposition_successful'):
            decomp = state['decomposition']

            fig, axes = plt.subplots(4, 1, figsize=(14, 10), dpi=CONFIG['dpi'])

            # Original
            axes[0].plot(series.index, series.values, color='#2E86AB')
            axes[0].set_ylabel('Original')
            axes[0].set_title('Time Series Decomposition', fontsize=14, fontweight='bold')
            axes[0].grid(True, alpha=0.3)

            # Trend
            axes[1].plot(decomp['trend'].index, decomp['trend'].values, color='#F18F01')
            axes[1].set_ylabel('Trend')
            axes[1].grid(True, alpha=0.3)

            # Seasonal
            axes[2].plot(decomp['seasonal'].index, decomp['seasonal'].values, color='#6A994E')
            axes[2].set_ylabel('Seasonal')
            axes[2].grid(True, alpha=0.3)

            # Residual
            axes[3].plot(decomp['residual'].index, decomp['residual'].values, color='#BC4749')
            axes[3].set_ylabel('Residual')
            axes[3].set_xlabel('Date')
            axes[3].grid(True, alpha=0.3)

            plt.tight_layout()
            decomp_path = 'decomposition_plot.png'
            plt.savefig(decomp_path)
            plt.close()
            plots['decomposition'] = decomp_path

            logger.info(f"✓ Decomposition plot saved: {decomp_path}")

        # 3. Residuals plot (if available)
        if state.get('residual_analysis'):
            test_data = state['test_data']
            model = state['trained_model']
            model_type = state['selected_model_type']

            if model_type in ['ARIMA', 'SARIMA']:
                predictions = model.forecast(steps=len(test_data))
            elif model_type == 'ExponentialSmoothing':
                predictions = model.forecast(steps=len(test_data))
            else:
                predictions = None

            if predictions is not None:
                residuals = test_data.values - predictions

                fig, axes = plt.subplots(1, 2, figsize=CONFIG['figure_size'], dpi=CONFIG['dpi'])

                # Residuals over time
                axes[0].plot(test_data.index, residuals, color='#BC4749', marker='o', linestyle='-', markersize=4)
                axes[0].axhline(y=0, color='black', linestyle='--', linewidth=1)
                axes[0].set_xlabel('Date')
                axes[0].set_ylabel('Residuals')
                axes[0].set_title('Residuals Over Time')
                axes[0].grid(True, alpha=0.3)

                # Residuals distribution
                axes[1].hist(residuals, bins=30, color='#BC4749', alpha=0.7, edgecolor='black')
                axes[1].set_xlabel('Residual Value')
                axes[1].set_ylabel('Frequency')
                axes[1].set_title('Residuals Distribution')
                axes[1].grid(True, alpha=0.3)

                plt.tight_layout()
                residuals_path = 'residuals_plot.png'
                plt.savefig(residuals_path)
                plt.close()
                plots['residuals'] = residuals_path

                logger.info(f"✓ Residuals plot saved: {residuals_path}")

        state['plots'] = plots
        state['processing_stage'] = 'visualizations_created'

        logger.info(f"✓ Generated {len(plots)} visualizations")

    except Exception as e:
        state['warnings'].append(f"Visualization generation failed: {str(e)}")
        logger.warning(f"⚠ Visualization failed: {e}")
        state['processing_stage'] = 'visualizations_created'

    return state

def output_formatter_node(state: TimeSeriesState) -> TimeSeriesState:
    """Format and save outputs"""
    logger.info("=" * 80)
    logger.info("STAGE 14: Output Formatting")
    logger.info("=" * 80)

    try:
        # 1. Save forecast results CSV
        forecast_df = pd.DataFrame({
            'date': state['forecast_dates'],
            'forecast': state['forecast_values'].values,
            'lower_95': state['confidence_intervals']['95_lower'],
            'upper_95': state['confidence_intervals']['95_upper']
        })
        forecast_df.to_csv('forecast_results.csv', index=False)
        logger.info("✓ Saved: forecast_results.csv")

        # 2. Save forecast report
        report = []
        report.append("="*80)
        report.append("TIME SERIES FORECASTING REPORT")
        report.append("="*80)
        report.append(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

        report.append("\n" + "-"*80)
        report.append("DATA SUMMARY")
        report.append("-"*80)
        profile = state['data_profile']
        report.append(f"Total Observations: {profile['total_observations']}")
        report.append(f"Date Range: {profile['date_range']['start']} to {profile['date_range']['end']}")
        report.append(f"Frequency: {profile['frequency']}")
        report.append(f"Missing Values: {profile['missing_values']} ({profile['missing_pct']:.2f}%)")

        report.append("\n" + "-"*80)
        report.append("STATISTICAL SUMMARY")
        report.append("-"*80)
        stats = profile['statistics']
        report.append(f"Mean: {stats['mean']:.2f}")
        report.append(f"Median: {stats['median']:.2f}")
        report.append(f"Std Dev: {stats['std']:.2f}")
        report.append(f"Min: {stats['min']:.2f}")
        report.append(f"Max: {stats['max']:.2f}")

        report.append("\n" + "-"*80)
        report.append("PATTERN ANALYSIS")
        report.append("-"*80)
        trend = state['trend_analysis']
        if trend.get('trend_detected'):
            report.append(f"Trend: {trend['trend_type'].upper()} (strength: {trend['trend_strength']:.3f})")
        else:
            report.append("Trend: No significant trend detected")

        seasonality = state['seasonality_analysis']
        if seasonality.get('seasonality_detected'):
            report.append(f"Seasonality: Detected (frequency: {seasonality['frequency']}, strength: {seasonality['strength']:.3f})")
        else:
            report.append("Seasonality: No significant seasonality detected")

        report.append("\n" + "-"*80)
        report.append("MODEL INFORMATION")
        report.append("-"*80)
        report.append(f"Model Type: {state['selected_model_type']}")
        report.append(f"Parameters: {json.dumps(state['model_parameters'], indent=2)}")

        if state.get('validation_metrics'):
            report.append("\n" + "-"*80)
            report.append("VALIDATION METRICS")
            report.append("-"*80)
            metrics = state['validation_metrics']
            report.append(f"MAE: {metrics['mae']:.2f}")
            report.append(f"RMSE: {metrics['rmse']:.2f}")
            report.append(f"MAPE: {metrics['mape']:.2f}%")

        report.append("\n" + "-"*80)
        report.append("FORECAST SUMMARY")
        report.append("-"*80)
        report.append(f"Horizon: {state['forecast_horizon']} periods")
        report.append(f"Forecast Range: {state['forecast_dates'][0]} to {state['forecast_dates'][-1]}")
        report.append(f"Mean Forecast: {state['forecast_values'].mean():.2f}")
        report.append(f"Min Forecast: {state['forecast_values'].min():.2f}")
        report.append(f"Max Forecast: {state['forecast_values'].max():.2f}")

        if state.get('pattern_insights'):
            report.append("\n" + "-"*80)
            report.append("PATTERN INSIGHTS")
            report.append("-"*80)
            for i, insight in enumerate(state['pattern_insights'], 1):
                report.append(f"{i}. {insight}")

        if state.get('forecast_insights'):
            report.append("\n" + "-"*80)
            report.append("FORECAST INSIGHTS")
            report.append("-"*80)
            for i, insight in enumerate(state['forecast_insights'], 1):
                report.append(f"{i}. {insight}")

        if state.get('recommendations'):
            report.append("\n" + "-"*80)
            report.append("RECOMMENDATIONS")
            report.append("-"*80)
            for i, rec in enumerate(state['recommendations'], 1):
                report.append(f"{i}. {rec}")

        report.append("\n" + "="*80)
        report.append("END OF REPORT")
        report.append("="*80)

        with open('forecast_report.txt', 'w') as f:
            f.write('\n'.join(report))
        logger.info("✓ Saved: forecast_report.txt")

        # 3. Save trained model
        with open('trained_model.pkl', 'wb') as f:
            pickle.dump(state['trained_model'], f)
        logger.info("✓ Saved: trained_model.pkl")

        # 4. Save model metadata
        metadata = {
            'model_type': state['selected_model_type'],
            'parameters': state['model_parameters'],
            'training_date': datetime.now().isoformat(),
            'data_profile': state['data_profile'],
            'validation_metrics': state.get('validation_metrics', {}),
            'forecast_horizon': state['forecast_horizon']
        }

        with open('model_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2, default=str)
        logger.info("✓ Saved: model_metadata.json")

        state['processing_stage'] = 'complete'

        logger.info("\n" + "="*80)
        logger.info("ALL OUTPUTS SAVED SUCCESSFULLY")
        logger.info("="*80)

    except Exception as e:
        state['errors'].append(f"Output formatting failed: {str(e)}")
        logger.error(f"✗ Output formatting failed: {e}")

    return state

# ============================================================================
# LANGGRAPH WORKFLOW
# ============================================================================

def create_forecaster_graph():
    """Create the LangGraph workflow"""

    workflow = StateGraph(TimeSeriesState)

    # Add nodes (renamed to avoid conflicts with state keys)
    workflow.add_node("load_file", file_loader_node)
    workflow.add_node("identify_columns", column_identifier_node)
    workflow.add_node("validate_data", data_validator_node)
    workflow.add_node("analyze_exploratory", exploratory_analysis_node)
    workflow.add_node("test_stationarity", stationarity_test_node)
    workflow.add_node("recommend_model", model_recommendation_node)
    workflow.add_node("select_model", model_selection_node)
    workflow.add_node("split_data", data_splitting_node)
    workflow.add_node("train_model", model_training_node)
    workflow.add_node("evaluate_model", model_evaluation_node)
    workflow.add_node("generate_forecast", forecasting_node)
    workflow.add_node("generate_insights", insights_generation_node)
    workflow.add_node("create_visualizations", visualization_node)
    workflow.add_node("format_output", output_formatter_node)

    # Define edges
    workflow.set_entry_point("load_file")
    workflow.add_edge("load_file", "identify_columns")
    workflow.add_edge("identify_columns", "validate_data")
    workflow.add_edge("validate_data", "analyze_exploratory")
    workflow.add_edge("analyze_exploratory", "test_stationarity")
    workflow.add_edge("test_stationarity", "recommend_model")
    workflow.add_edge("recommend_model", "select_model")
    workflow.add_edge("select_model", "split_data")
    workflow.add_edge("split_data", "train_model")
    workflow.add_edge("train_model", "evaluate_model")
    workflow.add_edge("evaluate_model", "generate_forecast")
    workflow.add_edge("generate_forecast", "generate_insights")
    workflow.add_edge("generate_insights", "create_visualizations")
    workflow.add_edge("create_visualizations", "format_output")
    workflow.add_edge("format_output", END)

    return workflow.compile()

# ============================================================================
# MAIN CLI INTERFACE
# ============================================================================

def main():
    """Main execution function"""
    print("\n" + "="*80)
    print("TIME SERIES FORECASTER AGENT")
    print("Powered by LangGraph & OpenAI")
    print("="*80)

    try:
        # Get file from user
        filepath = get_file_from_user()

        if not filepath or not Path(filepath).exists():
            print(f"\n❌ Error: File not found: {filepath}")
            return

        # Get forecast horizon
        print("\n" + "-"*80)
        horizon_input = input(f"Enter forecast horizon (days/periods) [default: {CONFIG['default_horizon']}]: ").strip()

        try:
            forecast_horizon = int(horizon_input) if horizon_input else CONFIG['default_horizon']
        except ValueError:
            forecast_horizon = CONFIG['default_horizon']
            print(f"Invalid input, using default: {forecast_horizon}")

        # Initialize state
        initial_state = TimeSeriesState(
            raw_data=None,
            filepath=filepath,
            date_column=None,
            value_column=None,
            forecast_horizon=forecast_horizon,
            frequency=None,
            data_profile=None,
            trend_analysis=None,
            seasonality_analysis=None,
            stationarity_test=None,
            decomposition=None,
            recommended_models=None,
            selected_model_type=None,
            model_parameters=None,
            train_data=None,
            test_data=None,
            trained_model=None,
            validation_metrics=None,
            residual_analysis=None,
            forecast_values=None,
            confidence_intervals=None,
            forecast_dates=None,
            pattern_insights=None,
            forecast_insights=None,
            recommendations=None,
            plots=None,
            processing_stage='initialized',
            errors=[],
            warnings=[],
            retry_count=0
        )

        # Create and run graph
        print("\n" + "="*80)
        print("STARTING ANALYSIS PIPELINE")
        print("="*80 + "\n")

        graph = create_forecaster_graph()
        final_state = graph.invoke(initial_state)

        # Display results
        print("\n" + "="*80)
        print("ANALYSIS COMPLETE")
        print("="*80)

        if final_state['errors']:
            print("\n❌ ERRORS:")
            for error in final_state['errors']:
                print(f"  - {error}")

        if final_state['warnings']:
            print("\n⚠️  WARNINGS:")
            for warning in final_state['warnings']:
                print(f"  - {warning}")

        if final_state['processing_stage'] == 'complete':
            print("\n✅ SUCCESS!")
            print("\nGenerated Files:")
            print("  1. forecast_results.csv - Forecast values with confidence intervals")
            print("  2. forecast_report.txt - Comprehensive analysis report")
            print("  3. trained_model.pkl - Trained forecasting model")
            print("  4. model_metadata.json - Model configuration and metadata")

            if final_state.get('plots'):
                print("\nGenerated Plots:")
                for plot_type, plot_path in final_state['plots'].items():
                    print(f"  - {plot_path}")

            # Display key insights
            if final_state.get('forecast_insights'):
                print("\n" + "-"*80)
                print("KEY FORECAST INSIGHTS:")
                print("-"*80)
                for i, insight in enumerate(final_state['forecast_insights'][:3], 1):
                    print(f"{i}. {insight}")

            if final_state.get('recommendations'):
                print("\n" + "-"*80)
                print("TOP RECOMMENDATIONS:")
                print("-"*80)
                for i, rec in enumerate(final_state['recommendations'][:3], 1):
                    print(f"{i}. {rec}")

        print("\n" + "="*80)
        print("Thank you for using TimeSeriesForecaster Agent!")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n\n⚠️  Process interrupted by user")
    except Exception as e:
        print(f"\n❌ Fatal error: {str(e)}")
        logger.exception("Fatal error in main execution")

if __name__ == "__main__":
    main()


TIME SERIES FORECASTER AGENT
Powered by LangGraph & OpenAI

FILE SELECTION

Enter the path to your time-series data file: /content/sample_timeseries_data.csv

--------------------------------------------------------------------------------
Enter forecast horizon (days/periods) [default: 30]: 15

STARTING ANALYSIS PIPELINE




ANALYSIS COMPLETE

⚠️  WARNINGS:
  - Model evaluation encountered issues: Input contains NaN.

✅ SUCCESS!

Generated Files:
  1. forecast_results.csv - Forecast values with confidence intervals
  2. forecast_report.txt - Comprehensive analysis report
  3. trained_model.pkl - Trained forecasting model
  4. model_metadata.json - Model configuration and metadata

Generated Plots:
  - forecast_plot.png
  - decomposition_plot.png

--------------------------------------------------------------------------------
KEY FORECAST INSIGHTS:
--------------------------------------------------------------------------------
1. The forecast for the next 15 days indicates a mean value of approximately 1423, which is slightly lower than the historical mean, suggesting a potential dip.
2. The minimum forecast value is projected at around 481, indicating possible low periods that should be monitored closely to understand underlying causes.
3. The maximum forecast value of approximately 1927 suggests potent